# Model Input Comparison: Temperature & Precipitation
## Evaluating Input Differences Between FSM2, seNorge, and SNOWPACK

This notebook compares the input forcing data (temperature and precipitation) used by the three snow models to understand how input uncertainty contributes to output differences in snow depth predictions.

In [26]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Model Data

In [27]:
# Load model datasets
fsm_ds = xr.open_dataset("../data/processed/fsm2_sd_all.nc")
senorge_ds = xr.open_dataset("../data/processed/senorge_all.nc")
snowpack_ds = xr.open_dataset("../data/processed/snowpack_all.nc")

print("FSM2 variables:")
print(list(fsm_ds.data_vars))
print("\nseNorge variables:")
print(list(senorge_ds.data_vars))
print("\nSNOWPACK variables:")
print(list(snowpack_ds.data_vars))

FSM2 variables:
['snow_depth', 'UTM_Zone_33']

seNorge variables:
['snow_depth', 'projection_info', 'UTM_Zone_33', 'snow_water_equivalent']

SNOWPACK variables:
['Qs', 'Ql', 'Qg', 'TSG', 'Qg0', 'Qr', 'dIntEnergySnow', 'meltFreezeEnergySnow', 'ColdContentSnow', 'OLWR', 'ILWR', 'LWR_net', 'RSWR', 'ISWR', 'Qw', 'pAlbedo', 'mAlbedo', 'ISWR_h', 'ISWR_dir', 'ISWR_diff', 'TA', 'TSS_mod', 'TSS_meas', 'T_bottom', 'RH', 'VW', 'VW_drift', 'DW', 'MS_Snow', 'HS_mod', 'HS_meas', 'hoar_size', 'wind_trans24', 'HN3', 'HN6', 'HN12', 'HN24', 'HN72_24', 'HNW24', 'ski_pen', 'SWE', 'MS_Water', 'MS_Water_Soil', 'MS_Ice_Soil', 'MS_Wind', 'MS_Rain', 'MS_SN_Runoff', 'MS_Surface_Mass_Flux', 'MS_Soil_Runoff', 'MS_Sublimation', 'MS_Evap', 'Sclass1', 'Sclass2', 'zSd', 'Sd', 'zSn', 'Sn', 'zSs', 'Ss', 'zS4', 'S4', 'zS5', 'S5']


## 2. Extract Temperature Data

In [28]:
# Note: Input forcing data (temperature, precipitation) is not stored in the processed files
# FSM2 and seNorge only contain aggregated snow_depth outputs
# SNOWPACK contains full simulation variables including temperature (TA) and energy terms

print("\nDATA AVAILABILITY:")
print(f"FSM2: {list(fsm_ds.data_vars)}")
print(f"seNorge: {list(senorge_ds.data_vars)}")
print(f"SNOWPACK has {len(snowpack_ds.data_vars)} variables")

# Extract available temperature data
snowpack_temp = None
try:
    snowpack_temp = snowpack_ds["TA"]  # TA is air temperature in SNOWPACK
    print(f"\n✓ SNOWPACK temperature (TA) found: {snowpack_temp.shape}")
except KeyError:
    print("✗ Temperature variable not found in SNOWPACK")

print("\n⚠️  Temperature forcing data not available in FSM2 and seNorge (only snow_depth outputs stored)")


DATA AVAILABILITY:
FSM2: ['snow_depth', 'UTM_Zone_33']
seNorge: ['snow_depth', 'projection_info', 'UTM_Zone_33', 'snow_water_equivalent']
SNOWPACK has 63 variables

✓ SNOWPACK temperature (TA) found: (4484, 6575)

⚠️  Temperature forcing data not available in FSM2 and seNorge (only snow_depth outputs stored)


## 3. Extract Precipitation Data

In [29]:
# Extract precipitation
# Note: FSM2 and seNorge don't have precipitation in processed files
# For SNOWPACK, use MS_Rain (mass of rain) as proxy

fsm_precip = None
senorge_precip = None
snowpack_precip = None

try:
    snowpack_precip = snowpack_ds["MS_Rain"]  # Mass of rain in SNOWPACK
    print(f"✓ SNOWPACK MS_Rain found: {snowpack_precip.shape}")
except KeyError:
    print("✗ MS_Rain variable not found in SNOWPACK")

print("\n⚠️  Precipitation (PSUM) forcing data not available in FSM2 and seNorge")

✓ SNOWPACK MS_Rain found: (4484, 6575)

⚠️  Precipitation (PSUM) forcing data not available in FSM2 and seNorge


## 4. Temperature Comparison

In [30]:
# If temperature data exists, compare them
if fsm_temp is not None and senorge_temp is not None:
    # Calculate seasonal mean temperatures
    fsm_tmean = float(fsm_temp.mean())
    sen_tmean = float(senorge_temp.mean())
    
    if snowpack_temp is not None:
        sp_tmean = float(snowpack_temp.mean())
    
    print(f"Mean Temperature across study period:")
    print(f"  FSM2: {fsm_tmean:.2f}°C")
    print(f"  seNorge: {sen_tmean:.2f}°C")
    if snowpack_temp is not None:
        print(f"  SNOWPACK: {sp_tmean:.2f}°C")
    
    # Temperature differences
    print(f"\nTemperature differences:")
    print(f"  FSM2 - seNorge: {fsm_tmean - sen_tmean:.2f}°C")
    if snowpack_temp is not None:
        print(f"  FSM2 - SNOWPACK: {fsm_tmean - sp_tmean:.2f}°C")
        print(f"  seNorge - SNOWPACK: {sen_tmean - sp_tmean:.2f}°C")

## 5. Precipitation Comparison

In [31]:
# If precipitation data exists, compare them
if fsm_precip is not None and senorge_precip is not None:
    # Calculate total precipitation
    fsm_ptotal = float(fsm_precip.sum())
    sen_ptotal = float(senorge_precip.sum())
    
    if snowpack_precip is not None:
        sp_ptotal = float(snowpack_precip.sum())
    
    print(f"Total Precipitation across study period:")
    print(f"  FSM2: {fsm_ptotal:.1f} mm")
    print(f"  seNorge: {sen_ptotal:.1f} mm")
    if snowpack_precip is not None:
        print(f"  SNOWPACK: {sp_ptotal:.1f} mm")
    
    # Precipitation differences
    print(f"\nPrecipitation differences:")
    pct_diff_fsm_sen = (fsm_ptotal - sen_ptotal) / sen_ptotal * 100
    print(f"  FSM2 vs seNorge: {pct_diff_fsm_sen:+.1f}%")
    if snowpack_precip is not None:
        pct_diff_fsm_sp = (fsm_ptotal - sp_ptotal) / sp_ptotal * 100
        pct_diff_sen_sp = (sen_ptotal - sp_ptotal) / sp_ptotal * 100
        print(f"  FSM2 vs SNOWPACK: {pct_diff_fsm_sp:+.1f}%")
        print(f"  seNorge vs SNOWPACK: {pct_diff_sen_sp:+.1f}%")

## 6. Temporal Evolution of Temperature

In [32]:
# Plot temporal evolution of temperature if available
if fsm_temp is not None and senorge_temp is not None:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Spatial mean temperature time series
    fsm_spatial_mean = fsm_temp.mean(dim=[d for d in fsm_temp.dims if d != 'time'])
    sen_spatial_mean = senorge_temp.mean(dim=[d for d in senorge_temp.dims if d != 'time'])
    
    ax = axes[0]
    fsm_spatial_mean.plot(ax=ax, label='FSM2', linewidth=2, marker='o', markersize=3)
    sen_spatial_mean.plot(ax=ax, label='seNorge', linewidth=2, marker='s', markersize=3)
    if snowpack_temp is not None:
        sp_spatial_mean = snowpack_temp.mean(dim=[d for d in snowpack_temp.dims if d != 'time'])
        sp_spatial_mean.plot(ax=ax, label='SNOWPACK', linewidth=2, marker='^', markersize=3)
    ax.set_ylabel('Mean Temperature (°C)', fontsize=11, fontweight='bold')
    ax.set_title('Spatial Mean Temperature Evolution', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Temperature differences
    ax = axes[1]
    temp_diff = fsm_spatial_mean - sen_spatial_mean
    ax.plot(temp_diff.time, temp_diff.values, label='FSM2 - seNorge', linewidth=2, marker='o', markersize=3)
    if snowpack_temp is not None:
        temp_diff_sp = fsm_spatial_mean - sp_spatial_mean
        ax.plot(temp_diff_sp.time, temp_diff_sp.values, label='FSM2 - SNOWPACK', linewidth=2, marker='^', markersize=3)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    ax.set_ylabel('Temperature Difference (°C)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Time', fontsize=11, fontweight='bold')
    ax.set_title('Temperature Differences Between Models', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../data/processed/temperature_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Figure saved to: ../data/processed/temperature_comparison.png")
else:
    print("Temperature data not available for comparison")

Temperature data not available for comparison


## 7. Temporal Evolution of Precipitation

In [33]:
# Plot temporal evolution of precipitation
if fsm_precip is not None and senorge_precip is not None:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Spatial mean precipitation time series
    fsm_spatial_precip = fsm_precip.mean(dim=[d for d in fsm_precip.dims if d != 'time'])
    sen_spatial_precip = senorge_precip.mean(dim=[d for d in senorge_precip.dims if d != 'time'])
    
    ax = axes[0]
    fsm_spatial_precip.plot(ax=ax, label='FSM2', linewidth=2, marker='o', markersize=3)
    sen_spatial_precip.plot(ax=ax, label='seNorge', linewidth=2, marker='s', markersize=3)
    if snowpack_precip is not None:
        sp_spatial_precip = snowpack_precip.mean(dim=[d for d in snowpack_precip.dims if d != 'time'])
        sp_spatial_precip.plot(ax=ax, label='SNOWPACK', linewidth=2, marker='^', markersize=3)
    ax.set_ylabel('Daily Precipitation (mm)', fontsize=11, fontweight='bold')
    ax.set_title('Spatial Mean Precipitation Evolution', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Relative differences
    ax = axes[1]
    # Use percentage difference where seNorge is reference
    precip_pct_diff = (fsm_spatial_precip - sen_spatial_precip) / sen_spatial_precip * 100
    precip_pct_diff = precip_pct_diff.where(np.isfinite(precip_pct_diff), np.nan)
    ax.plot(precip_pct_diff.time, precip_pct_diff.values, label='(FSM2 - seNorge)/seNorge (%)', linewidth=2, marker='o', markersize=3)
    if snowpack_precip is not None:
        precip_pct_diff_sp = (fsm_spatial_precip - sp_spatial_precip) / sp_spatial_precip * 100
        precip_pct_diff_sp = precip_pct_diff_sp.where(np.isfinite(precip_pct_diff_sp), np.nan)
        ax.plot(precip_pct_diff_sp.time, precip_pct_diff_sp.values, label='(FSM2 - SNOWPACK)/SNOWPACK (%)', linewidth=2, marker='^', markersize=3)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    ax.set_ylabel('Relative Difference (%)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Time', fontsize=11, fontweight='bold')
    ax.set_title('Relative Precipitation Differences Between Models', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../data/processed/precipitation_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Figure saved to: ../data/processed/precipitation_comparison.png")
else:
    print("Precipitation data not available for comparison")

Precipitation data not available for comparison


## 8. Spatial Variability of Inputs

In [34]:
# Calculate spatial statistics of inputs
input_stats = []

if fsm_temp is not None:
    input_stats.append({
        'Model': 'FSM2',
        'Variable': 'Temperature',
        'Spatial Min': float(fsm_temp.min()),
        'Spatial Max': float(fsm_temp.max()),
        'Spatial Std': float(fsm_temp.std()),
        'Mean': float(fsm_temp.mean())
    })

if senorge_temp is not None:
    input_stats.append({
        'Model': 'seNorge',
        'Variable': 'Temperature',
        'Spatial Min': float(senorge_temp.min()),
        'Spatial Max': float(senorge_temp.max()),
        'Spatial Std': float(senorge_temp.std()),
        'Mean': float(senorge_temp.mean())
    })

if snowpack_temp is not None:
    input_stats.append({
        'Model': 'SNOWPACK',
        'Variable': 'Temperature',
        'Spatial Min': float(snowpack_temp.min()),
        'Spatial Max': float(snowpack_temp.max()),
        'Spatial Std': float(snowpack_temp.std()),
        'Mean': float(snowpack_temp.mean())
    })

if fsm_precip is not None:
    input_stats.append({
        'Model': 'FSM2',
        'Variable': 'Precipitation',
        'Spatial Min': float(fsm_precip.min()),
        'Spatial Max': float(fsm_precip.max()),
        'Spatial Std': float(fsm_precip.std()),
        'Mean': float(fsm_precip.mean())
    })

if senorge_precip is not None:
    input_stats.append({
        'Model': 'seNorge',
        'Variable': 'Precipitation',
        'Spatial Min': float(senorge_precip.min()),
        'Spatial Max': float(senorge_precip.max()),
        'Spatial Std': float(senorge_precip.std()),
        'Mean': float(senorge_precip.mean())
    })

if snowpack_precip is not None:
    input_stats.append({
        'Model': 'SNOWPACK',
        'Variable': 'Precipitation',
        'Spatial Min': float(snowpack_precip.min()),
        'Spatial Max': float(snowpack_precip.max()),
        'Spatial Std': float(snowpack_precip.std()),
        'Mean': float(snowpack_precip.mean())
    })

input_df = pd.DataFrame(input_stats)
print("\nSpatial Statistics of Model Inputs:")
print(input_df.to_string(index=False))

# Save to CSV
input_df.to_csv('../data/processed/model_input_statistics.csv', index=False)
print("\nStatistics saved to: ../data/processed/model_input_statistics.csv")


Spatial Statistics of Model Inputs:
   Model      Variable  Spatial Min  Spatial Max  Spatial Std      Mean
SNOWPACK   Temperature      -34.709       25.124     6.838467 -2.916506
SNOWPACK Precipitation        0.000       20.246     0.367489  0.055990

Statistics saved to: ../data/processed/model_input_statistics.csv


## 9. Correlation Between Inputs and Output Differences

In [35]:
# Load validation results to correlate input differences with output differences
validation_results = pd.read_csv('../data/processed/multi_station_validation_results.csv')

# Calculate mean input differences
if fsm_temp is not None and senorge_temp is not None:
    temp_diff_mean = float(fsm_temp.mean() - senorge_temp.mean())
    print(f"Mean Temperature Difference (FSM2 - seNorge): {temp_diff_mean:.2f}°C")

if fsm_precip is not None and senorge_precip is not None:
    precip_diff_mean = float(fsm_precip.sum() - senorge_precip.sum())
    print(f"Total Precipitation Difference (FSM2 - seNorge): {precip_diff_mean:.1f} mm")

# Compare with snow depth output differences
validation_results['FSM2_seNorge_RMSE_Diff'] = validation_results['FSM2_RMSE'] - validation_results['seNorge_RMSE']
validation_results['FSM2_seNorge_Bias_Diff'] = abs(validation_results['FSM2_Bias']) - abs(validation_results['seNorge_Bias'])

print(f"\nMean RMSE Difference (FSM2 - seNorge): {validation_results['FSM2_seNorge_RMSE_Diff'].mean():.4f} m")
print(f"Mean Bias Difference (FSM2 - seNorge): {validation_results['FSM2_seNorge_Bias_Diff'].mean():.4f} m")

print("\n" + "="*80)
print("INTERPRETATION:")
print("="*80)
print("Input differences contribute to output differences in snow depth.")
print("Temperature impacts: Affects the snow/rain threshold and melt processes.")
print("Precipitation impacts: Directly affects snow accumulation.")


Mean RMSE Difference (FSM2 - seNorge): -0.1447 m
Mean Bias Difference (FSM2 - seNorge): -0.0793 m

INTERPRETATION:
Input differences contribute to output differences in snow depth.
Temperature impacts: Affects the snow/rain threshold and melt processes.
Precipitation impacts: Directly affects snow accumulation.


## 10. Summary Statistics Table

In [36]:
# Create comprehensive summary
print("\n" + "="*100)
print("COMPREHENSIVE INPUT COMPARISON SUMMARY")
print("="*100)

print("\n1. TEMPERATURE ANALYSIS:")
if fsm_temp is not None and senorge_temp is not None:
    print(f"   - FSM2 mean:       {float(fsm_temp.mean()):.2f}°C")
    print(f"   - seNorge mean:    {float(senorge_temp.mean()):.2f}°C")
    if snowpack_temp is not None:
        print(f"   - SNOWPACK mean:   {float(snowpack_temp.mean()):.2f}°C")
    print(f"   - Implication: Temperature differences affect phase partitioning and melt rates")
else:
    print("   - Temperature data not available in datasets")

print("\n2. PRECIPITATION ANALYSIS:")
if fsm_precip is not None and senorge_precip is not None:
    print(f"   - FSM2 total:      {float(fsm_precip.sum()):.1f} mm")
    print(f"   - seNorge total:   {float(senorge_precip.sum()):.1f} mm")
    if snowpack_precip is not None:
        print(f"   - SNOWPACK total:  {float(snowpack_precip.sum()):.1f} mm")
    print(f"   - Implication: Precipitation differences directly affect snow accumulation")
else:
    print("   - Precipitation data not available in datasets")

print("\n3. SPATIAL VARIABILITY:")
print(f"   See model_input_statistics.csv for detailed spatial ranges")

print("\n4. LINKS TO OUTPUT DIFFERENCES:")
print(f"   - Mean FSM2 RMSE:     {validation_results['FSM2_RMSE'].mean():.4f} m")
print(f"   - Mean seNorge RMSE:  {validation_results['seNorge_RMSE'].mean():.4f} m")
if 'SNOWPACK_RMSE' in validation_results.columns:
    print(f"   - Mean SNOWPACK RMSE: {validation_results['SNOWPACK_RMSE'].mean():.4f} m")
else:
    print("   - SNOWPACK RMSE: Not available (run notebook 10 first to generate all metrics)")
print("\n" + "="*100)


COMPREHENSIVE INPUT COMPARISON SUMMARY

1. TEMPERATURE ANALYSIS:
   - Temperature data not available in datasets

2. PRECIPITATION ANALYSIS:
   - Precipitation data not available in datasets

3. SPATIAL VARIABILITY:
   See model_input_statistics.csv for detailed spatial ranges

4. LINKS TO OUTPUT DIFFERENCES:
   - Mean FSM2 RMSE:     1.8426 m
   - Mean seNorge RMSE:  1.9872 m
   - SNOWPACK RMSE: Not available (run notebook 10 first to generate all metrics)



In [37]:

print("FSM2 variables:")
print(list(fsm_ds.data_vars))
print("\nseNorge variables:")
print(list(senorge_ds.data_vars))
print("\nSNOWPACK variables:")
print(list(snowpack_ds.data_vars))

FSM2 variables:
['snow_depth', 'UTM_Zone_33']

seNorge variables:
['snow_depth', 'projection_info', 'UTM_Zone_33', 'snow_water_equivalent']

SNOWPACK variables:
['Qs', 'Ql', 'Qg', 'TSG', 'Qg0', 'Qr', 'dIntEnergySnow', 'meltFreezeEnergySnow', 'ColdContentSnow', 'OLWR', 'ILWR', 'LWR_net', 'RSWR', 'ISWR', 'Qw', 'pAlbedo', 'mAlbedo', 'ISWR_h', 'ISWR_dir', 'ISWR_diff', 'TA', 'TSS_mod', 'TSS_meas', 'T_bottom', 'RH', 'VW', 'VW_drift', 'DW', 'MS_Snow', 'HS_mod', 'HS_meas', 'hoar_size', 'wind_trans24', 'HN3', 'HN6', 'HN12', 'HN24', 'HN72_24', 'HNW24', 'ski_pen', 'SWE', 'MS_Water', 'MS_Water_Soil', 'MS_Ice_Soil', 'MS_Wind', 'MS_Rain', 'MS_SN_Runoff', 'MS_Surface_Mass_Flux', 'MS_Soil_Runoff', 'MS_Sublimation', 'MS_Evap', 'Sclass1', 'Sclass2', 'zSd', 'Sd', 'zSn', 'Sn', 'zSs', 'Ss', 'zS4', 'S4', 'zS5', 'S5']
